In [ ]:
import spare_seq.correlation as corr
import spare_seq.visualization as viz

# 1. Configuration
samples = ["49", "50", "51", "52", "53", "54"]
targets = ["ENSMUSG00000027951", "ENSMUSG00000020262", "ENSMUSG00000052551"]

H5AD_DIR = "PATH/TO/YOUR/GENE_H5AD_DIR"
CLUSTER_DIR = "PATH/TO/YOUR/CLUSTER_TXT_DIR"
FILTERED_MAT_DIR = "PATH/TO/YOUR/FILTERED_MATRICES" # Results from Workflow 1
OUTPUT_DIR = "./correlation_results"

# 2. Extract, sum, and normalize (log2(x+1)) target gene expression
corr.extract_and_normalize_genes(
    h5ad_dir=H5AD_DIR,
    cluster_dir=CLUSTER_DIR,
    save_dir=f"{OUTPUT_DIR}/gene_processed",
    sample_ids=samples,
    target_genes=targets
)

# 3. Reconstruct site union and calculate editing Ratio (G / (G+A+1))
corr.calculate_ratio_matrix(
    input_matrices_dir=FILTERED_MAT_DIR,
    save_dir=f"{OUTPUT_DIR}/ratio_matrices",
    sample_ids=samples
)

# 4. Run Pearson & Spearman correlation analysis 
corr.run_correlations(
    ratio_dir=f"{OUTPUT_DIR}/ratio_matrices",
    gene_dir=f"{OUTPUT_DIR}/gene_processed",
    output_dir=f"{OUTPUT_DIR}/raw_correlations",
    sample_ids=samples
)

# 5. Filter significant sites (Spearman Rho > 0.3, Pearson P < 0.05)
corr.filter_correlation_results(
    input_dir=f"{OUTPUT_DIR}/raw_correlations",
    output_dir=f"{OUTPUT_DIR}/filtered_significance",
    sample_ids=samples,
    spearman_rho=0.3,
    pearson_p=0.05
)

# 6. Visualize: Grouped bar plots for chromosome distribution
viz.plot_correlation_chr_distribution(
    filtered_dir=f"{OUTPUT_DIR}/filtered_significance",
    sample_ids=samples,
    save_dir=f"{OUTPUT_DIR}/plots"
)


# Configuration (Assuming you've run the correlation calculations)
samples = ["49", "50", "51", "52", "53", "54"]
RAW_CORR_DIR = "PATH/TO/YOUR/OUTPUT_DIR/raw_correlations"
BOXPLOT_OUT_DIR = "PATH/TO/YOUR/OUTPUT_DIR/BoxPlots_Correlation"

# 7. Generate Boxplots for ADAR Correlation
# By default, it plots both 'Spearman_Rho' and 'Pearson_R'. 
# It will automatically create sub-folders for each metric.
viz.plot_correlation_boxplots(
    input_dir=RAW_CORR_DIR,
    output_base_dir=BOXPLOT_OUT_DIR,
    sample_ids=samples,
    metrics=['Spearman_Rho', 'Pearson_R'] 
)